<a href="https://colab.research.google.com/github/gabriel5imas/APSprojeto/blob/main/APS_7_semestre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
if 'google.colab' in sys.modules:
  !pip install flask-cors
from flask import Flask, jsonify, request
from flask_cors import CORS
import time
import uuid

app = Flask(__name__)
CORS(app)

# Tempo em segundos para simular o escalonamento (representa 4 horas na lógica real)
TEMPO_ESCALONAMENTO = 30

# Base de dados em memória
sugestoes = [
    {
        "id": str(uuid.uuid4()),
        "fruta": "Banana Prata",
        "confianca": 92,
        "setor": "Gôndola 3",
        "timestamp": time.time() - 35,  # já passou do tempo de escalonamento
        "status": "PENDENTE",
        "desconto_sugerido": 30,
        "respondido": False,
        "tempo_resposta": None
    },
    {
        "id": str(uuid.uuid4()),
        "fruta": "Mamão Papaia",
        "confianca": 88,
        "setor": "Gôndola 1",
        "timestamp": time.time() - 10,
        "status": "PENDENTE",
        "desconto_sugerido": 25,
        "respondido": False,
        "tempo_resposta": None
    },
    {
        "id": str(uuid.uuid4()),
        "fruta": "Banana Nanica",
        "confianca": 95,
        "setor": "Gôndola 5",
        "timestamp": time.time() - 5,
        "status": "PENDENTE",
        "desconto_sugerido": 35,
        "respondido": False,
        "tempo_resposta": None
    },
    {
        "id": str(uuid.uuid4()),
        "fruta": "Maçã Fuji",
        "confianca": 79,
        "setor": "Gôndola 2",
        "timestamp": time.time() - 60,
        "status": "PENDENTE",
        "desconto_sugerido": 20,
        "respondido": False,
        "tempo_resposta": None
    },
    {
        "id": str(uuid.uuid4()),
        "fruta": "Banana Prata",
        "confianca": 91,
        "setor": "Gôndola 4",
        "timestamp": time.time() - 120,
        "status": "EXECUTADA",
        "desconto_sugerido": 30,
        "respondido": True,
        "tempo_resposta": 45
    },
]

def calcular_status(sugestao):
    """Atualiza o status com base no tempo decorrido desde a detecção."""
    if sugestao["respondido"]:
        return sugestao["status"]
    elapsed = time.time() - sugestao["timestamp"]
    if elapsed >= TEMPO_ESCALONAMENTO:
        return "RISCO_IMINENTE"
    return "PENDENTE"

def tempo_restante(sugestao):
    """Retorna os segundos restantes antes do escalonamento."""
    if sugestao["respondido"]:
        return 0
    elapsed = time.time() - sugestao["timestamp"]
    restante = TEMPO_ESCALONAMENTO - elapsed
    return max(0, round(restante))

@app.route("/api/sugestoes", methods=["GET"])
def get_sugestoes():
    resultado = []
    pendentes = 0
    total = len(sugestoes)

    for s in sugestoes:
        status_atual = calcular_status(s)
        if not s["respondido"]:
            s["status"] = status_atual
            pendentes += 1

        resultado.append({
            "id": s["id"],
            "fruta": s["fruta"],
            "confianca": s["confianca"],
            "setor": s["setor"],
            "timestamp": s["timestamp"],
            "status": status_atual,
            "desconto_sugerido": s["desconto_sugerido"],
            "respondido": s["respondido"],
            "tempo_resposta": s["tempo_resposta"],
            "segundos_restantes": tempo_restante(s)
        })

    # Ordenação: RISCO_IMINENTE primeiro, depois PENDENTE, depois respondidos
    ordem = {"RISCO_IMINENTE": 0, "PENDENTE": 1, "EXECUTADA": 2, "DESCARTADA": 3}
    resultado.sort(key=lambda x: ordem.get(x["status"], 99))

    return jsonify({
        "sugestoes": resultado,
        "metricas": {
            "total_deteccoes": total + 137,  # simula histórico maior
            "alertas_pendentes": pendentes,
            "taxa_efetividade": round(
                (len([s for s in sugestoes if s["status"] == "EXECUTADA"]) / max(total, 1)) * 100 + 70, 1
            )
        }
    })

@app.route("/api/responder", methods=["POST"])
def responder():
    dados = request.get_json()
    sugestao_id = dados.get("id")
    acao = dados.get("acao")  # "EXECUTADA" ou "DESCARTADA"

    if not sugestao_id or acao not in ("EXECUTADA", "DESCARTADA"):
        return jsonify({"erro": "Dados inválidos"}), 400

    for s in sugestoes:
        if s["id"] == sugestao_id:
            if s["respondido"]:
                return jsonify({"erro": "Sugestão já respondida"}), 409
            tempo_decorrido = round(time.time() - s["timestamp"])
            s["status"] = acao
            s["respondido"] = True
            s["tempo_resposta"] = tempo_decorrido
            return jsonify({
                "mensagem": f"Ação '{acao}' registrada com sucesso.",
                "tempo_resposta_segundos": tempo_decorrido
            })

    return jsonify({"erro": "Sugestão não encontrada"}), 404

@app.route("/api/log", methods=["GET"])
def get_log():
    import datetime
    now = datetime.datetime.now().strftime("%H:%M:%S")
    logs = [
        f"[{now}] INFO  Frame processado com sucesso — latência 1.8s",
        f"[{now}] DB    Inserção em tb_analises concluída (120ms)",
        f"[{now}] INFO  YOLOv8n Active | Confiança threshold: 0.70",
        f"[{now}] INFO  OpenCV Stream Active | Wi-Fi Local Network",
        f"[{now}] INFO  FastAPI Server Running — porta 5000",
    ]
    return jsonify({"logs": logs})

if __name__ == "__main__":
    app.run(debug=True, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [4]:
from IPython.display import HTML
HTML("""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>FreshGuard — Dashboard</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

    * { font-family: 'Inter', sans-serif; }

    body { background-color: #0f1117; }

    .sidebar-item {
      display: flex;
      align-items: center;
      gap: 12px;
      padding: 10px 16px;
      border-radius: 8px;
      color: #9ca3af;
      cursor: pointer;
      transition: all 0.2s;
      font-size: 14px;
      font-weight: 500;
    }
    .sidebar-item:hover { background: #1e2130; color: #f3f4f6; }
    .sidebar-item.active { background: #1e2130; color: #34d399; }

    .metric-card {
      background: #1a1d2e;
      border: 1px solid #2d3148;
      border-radius: 12px;
      padding: 20px 24px;
    }

    .alert-card {
      background: #1a1d2e;
      border: 1px solid #2d3148;
      border-radius: 10px;
      padding: 16px 20px;
      transition: all 0.3s ease;
    }
    .alert-card:hover { border-color: #4b5280; }
    .alert-card.risco { border-left: 3px solid #ef4444; }
    .alert-card.pendente { border-left: 3px solid #f59e0b; }
    .alert-card.executada { border-left: 3px solid #34d399; opacity: 0.6; }
    .alert-card.descartada { border-left: 3px solid #6b7280; opacity: 0.5; }

    .badge-pendente {
      background: rgba(245,158,11,0.15);
      color: #f59e0b;
      border: 1px solid rgba(245,158,11,0.3);
      padding: 2px 10px;
      border-radius: 20px;
      font-size: 11px;
      font-weight: 600;
      letter-spacing: 0.5px;
    }
    .badge-risco {
      background: rgba(239,68,68,0.15);
      color: #ef4444;
      border: 1px solid rgba(239,68,68,0.3);
      padding: 2px 10px;
      border-radius: 20px;
      font-size: 11px;
      font-weight: 600;
      letter-spacing: 0.5px;
      animation: pulse-red 1.5s infinite;
    }
    .badge-executada {
      background: rgba(52,211,153,0.15);
      color: #34d399;
      border: 1px solid rgba(52,211,153,0.3);
      padding: 2px 10px;
      border-radius: 20px;
      font-size: 11px;
      font-weight: 600;
    }
    .badge-descartada {
      background: rgba(107,114,128,0.15);
      color: #9ca3af;
      border: 1px solid rgba(107,114,128,0.3);
      padding: 2px 10px;
      border-radius: 20px;
      font-size: 11px;
      font-weight: 600;
    }

    @keyframes pulse-red {
      0%, 100% { opacity: 1; }
      50% { opacity: 0.6; }
    }

    .log-terminal {
      background: #0a0c12;
      border: 1px solid #2d3148;
      border-radius: 10px;
      font-family: 'Courier New', monospace;
      font-size: 12px;
      padding: 16px;
      height: 130px;
      overflow-y: auto;
      color: #4ade80;
    }
    .log-terminal::-webkit-scrollbar { width: 4px; }
    .log-terminal::-webkit-scrollbar-track { background: #0a0c12; }
    .log-terminal::-webkit-scrollbar-thumb { background: #2d3148; border-radius: 2px; }

    .confidence-bar-bg {
      background: #2d3148;
      border-radius: 4px;
      height: 5px;
      width: 80px;
    }
    .confidence-bar-fill {
      height: 5px;
      border-radius: 4px;
      background: linear-gradient(90deg, #34d399, #10b981);
    }

    .btn-action {
      padding: 5px 14px;
      border-radius: 6px;
      font-size: 12px;
      font-weight: 600;
      cursor: pointer;
      transition: all 0.15s;
      border: none;
    }
    .btn-desconto {
      background: rgba(52,211,153,0.15);
      color: #34d399;
      border: 1px solid rgba(52,211,153,0.3);
    }
    .btn-desconto:hover { background: rgba(52,211,153,0.3); }
    .btn-ignorar {
      background: rgba(107,114,128,0.15);
      color: #9ca3af;
      border: 1px solid rgba(107,114,128,0.2);
    }
    .btn-ignorar:hover { background: rgba(107,114,128,0.3); }

    .status-dot {
      width: 8px; height: 8px;
      border-radius: 50%;
      display: inline-block;
    }
    .dot-green { background: #34d399; box-shadow: 0 0 6px #34d399; }
    .dot-yellow { background: #f59e0b; box-shadow: 0 0 6px #f59e0b; }

    .counter-badge {
      background: rgba(239,68,68,0.2);
      color: #ef4444;
      border: 1px solid rgba(239,68,68,0.3);
      padding: 1px 7px;
      border-radius: 4px;
      font-size: 11px;
      font-weight: 700;
    }

    .toast {
      position: fixed;
      bottom: 24px;
      right: 24px;
      background: #1a1d2e;
      border: 1px solid #34d399;
      color: #34d399;
      padding: 12px 20px;
      border-radius: 8px;
      font-size: 13px;
      font-weight: 500;
      z-index: 9999;
      opacity: 0;
      transform: translateY(10px);
      transition: all 0.3s ease;
    }
    .toast.show { opacity: 1; transform: translateY(0); }
    .toast.error { border-color: #ef4444; color: #ef4444; }

    #feed-lista { display: flex; flex-direction: column; gap: 10px; }
  </style>
</head>
<body class="text-gray-100 min-h-screen flex">

  <!-- SIDEBAR -->
  <aside class="w-56 min-h-screen flex flex-col py-6 px-4 border-r border-gray-800" style="background:#13151f;">
    <div class="mb-8 px-2">
      <div class="flex items-center gap-2 mb-1">
        <span class="text-green-400 text-xl">🌿</span>
        <span class="text-white font-bold text-lg tracking-tight">FreshGuard</span>
      </div>
      <span class="text-xs text-gray-500 pl-7">v1.0 — TCC1 UNIP</span>
    </div>

    <nav class="flex flex-col gap-1">
      <div class="sidebar-item active">
        <span>📊</span> Dashboard
      </div>
      <div class="sidebar-item">
        <span>🔔</span> Alertas
      </div>
      <div class="sidebar-item">
        <span>📋</span> Relatórios
      </div>
      <div class="sidebar-item">
        <span>⚙️</span> Configurações
      </div>
    </nav>

    <div class="mt-auto">
      <div class="metric-card p-3">
        <div class="text-xs text-gray-500 mb-2 font-medium uppercase tracking-wide">Status do Sistema</div>
        <div class="flex items-center gap-2 text-xs text-gray-300 mb-1">
          <span class="status-dot dot-green"></span> YOLOv8n Ativo
        </div>
        <div class="flex items-center gap-2 text-xs text-gray-300 mb-1">
          <span class="status-dot dot-green"></span> SQLite Conectado
        </div>
        <div class="flex items-center gap-2 text-xs text-gray-300">
          <span class="status-dot dot-green"></span> Wi-Fi Local
        </div>
      </div>
    </div>
  </aside>

  <!-- MAIN -->
  <div class="flex-1 flex flex-col">

    <!-- HEADER -->
    <header class="flex items-center justify-between px-8 py-4 border-b border-gray-800" style="background:#13151f;">
      <div>
        <h1 class="text-white font-bold text-xl">Dashboard de Detecção</h1>
        <p class="text-gray-500 text-xs mt-0.5">Monitoramento de qualidade — Hortifrúti</p>
      </div>
      <div class="flex items-center gap-3">
        <div class="text-right">
          <div class="text-sm font-semibold text-white">Gerente da Loja</div>
          <div class="text-xs text-gray-500">Setor: Hortifrúti</div>
        </div>
        <div class="w-9 h-9 rounded-full bg-green-500 flex items-center justify-center text-white font-bold text-sm">G</div>
        <div class="flex items-center gap-1.5 ml-2 text-xs text-gray-400">
          <span class="status-dot dot-green"></span>
          <span id="last-update">Aguardando...</span>
        </div>
      </div>
    </header>

    <!-- CONTENT -->
    <main class="flex-1 p-8 overflow-y-auto">

      <!-- METRIC CARDS -->
      <div class="grid grid-cols-3 gap-5 mb-8">

        <div class="metric-card">
          <div class="flex items-center justify-between mb-3">
            <span class="text-xs text-gray-500 uppercase tracking-wide font-medium">Total de Detecções</span>
            <span class="text-2xl">📦</span>
          </div>
          <div id="metric-total" class="text-4xl font-bold text-white">—</div>
          <div class="text-xs text-gray-500 mt-1">Turno atual</div>
        </div>

        <div class="metric-card" style="border-color: rgba(245,158,11,0.3);">
          <div class="flex items-center justify-between mb-3">
            <span class="text-xs text-gray-500 uppercase tracking-wide font-medium">Alertas Pendentes</span>
            <span class="text-2xl">⚠️</span>
          </div>
          <div id="metric-alertas" class="text-4xl font-bold text-yellow-400">—</div>
          <div class="text-xs text-gray-500 mt-1">Requerem ação imediata</div>
        </div>

        <div class="metric-card" style="border-color: rgba(52,211,153,0.3);">
          <div class="flex items-center justify-between mb-3">
            <span class="text-xs text-gray-500 uppercase tracking-wide font-medium">Taxa de Efetividade</span>
            <span class="text-2xl">🎯</span>
          </div>
          <div id="metric-efetividade" class="text-4xl font-bold text-green-400">—</div>
          <div class="text-xs text-gray-500 mt-1">Ações concluídas vs. geradas</div>
        </div>

      </div>

      <!-- FEED + LOG -->
      <div class="grid grid-cols-5 gap-6">

        <!-- FEED DE ALERTAS (3/5) -->
        <div class="col-span-3">
          <div class="flex items-center justify-between mb-4">
            <h2 class="text-white font-semibold text-base">Fila de Sugestões Comerciais</h2>
            <span class="text-xs text-gray-500">Atualiza a cada 5s</span>
          </div>
          <div id="feed-lista">
            <div class="text-gray-500 text-sm text-center py-8">Carregando detecções...</div>
          </div>
        </div>

        <!-- PAINEL DIREITO (2/5) -->
        <div class="col-span-2 flex flex-col gap-5">

          <!-- STATUS CÂMERAS -->
          <div class="metric-card">
            <div class="text-xs text-gray-500 uppercase tracking-wide font-medium mb-3">Status das Câmeras</div>
            <div class="flex items-center justify-between py-2 border-b border-gray-800">
              <div class="flex items-center gap-2 text-sm text-gray-300">
                <span class="status-dot dot-green"></span>
                <span>Cam-01 / Gôndola 3</span>
              </div>
              <span class="text-xs text-green-400 font-medium">Online</span>
            </div>
            <div class="flex items-center justify-between py-2">
              <div class="flex items-center gap-2 text-sm text-gray-300">
                <span class="status-dot dot-yellow"></span>
                <span>Cam-02 / Gôndola 1</span>
              </div>
              <span class="text-xs text-yellow-400 font-medium">Instável</span>
            </div>
          </div>

          <!-- DESPERDÍCIO EVITADO -->
          <div class="metric-card">
            <div class="text-xs text-gray-500 uppercase tracking-wide font-medium mb-3">Desperdício Evitado (est.)</div>
            <div class="flex items-end gap-2">
              <span class="text-3xl font-bold text-green-400" id="desperdicio-kg">2.4</span>
              <span class="text-gray-500 text-sm mb-1">kg neste turno</span>
            </div>
            <div class="flex gap-1 mt-3" id="mini-grafico">
              <!-- Barras geradas via JS -->
            </div>
            <div class="text-xs text-gray-600 mt-1">Últimas 8 classificações MADURA respondidas</div>
          </div>

          <!-- LOG TERMINAL -->
          <div>
            <div class="text-xs text-gray-500 uppercase tracking-wide font-medium mb-2">Python Server Log [FastAPI]</div>
            <div class="log-terminal" id="log-terminal">
              <div class="text-green-500">Aguardando logs do servidor...</div>
            </div>
          </div>

        </div>
      </div>
    </main>
  </div>

  <!-- TOAST -->
  <div class="toast" id="toast"></div>

  <script>
    const API_BASE = "http://localhost:5000";
    let dadosAtuais = [];

    // --- Utilitários ---

    function formatTimestamp(ts) {
      const d = new Date(ts * 1000);
      return d.toLocaleTimeString("pt-BR", { hour: "2-digit", minute: "2-digit", second: "2-digit" });
    }

    function showToast(msg, tipo = "success") {
      const el = document.getElementById("toast");
      el.textContent = msg;
      el.className = `toast show${tipo === "error" ? " error" : ""}`;
      setTimeout(() => { el.classList.remove("show"); }, 3000);
    }

    function getBadgeHTML(status, segundos) {
      if (status === "RISCO_IMINENTE") {
        return `<span class="badge-risco">⚡ RISCO IMINENTE</span>\n                <span class="counter-badge ml-1">+${Math.abs(segundos)}s atraso</span>`;
      }
      if (status === "PENDENTE") {
        return `<span class="badge-pendente">⏳ PENDENTE</span>\n                <span class="text-xs text-gray-500 ml-1">${segundos}s restantes</span>`;
      }
      if (status === "EXECUTADA") return `<span class="badge-executada">✅ EXECUTADA</span>`;
      if (status === "DESCARTADA") return `<span class="badge-descartada">— IGNORADA</span>`;
      return `<span class="badge-pendente">${status}</span>`;
    }

    function getCardClass(status) {
      if (status === "RISCO_IMINENTE") return "alert-card risco";
      if (status === "PENDENTE") return "alert-card pendente";
      if (status === "EXECUTADA") return "alert-card executada";
      return "alert-card descartada";
    }

    function getFrutaEmoji(fruta) {
      const f = fruta.toLowerCase();
      if (f.includes("banana")) return "🍌";
      if (f.includes("maçã") || f.includes("maca")) return "🍎";
      if (f.includes("mamão") || f.includes("mamao")) return "🍈";
      if (f.includes("uva")) return "🍇";
      return "🍑";
    }

    // --- Renderização do Feed ---

    function renderFeed(sugestoes) {
      const lista = document.getElementById("feed-lista");

      if (!sugestoes || sugestoes.length === 0) {
        lista.innerHTML = `<div class="text-gray-500 text-sm text-center py-8">Nenhuma sugestão ativa no momento.</div>`;
        return;
      }

      lista.innerHTML = sugestoes.map(s => `
        <div class="${getCardClass(s.status)}" id="card-${s.id}">
          <div class="flex items-start justify-between gap-3">

            <div class="flex items-center gap-3 flex-1 min-w-0">
              <span class="text-2xl">${getFrutaEmoji(s.fruta)}</span>
              <div class="min-w-0">
                <div class="flex items-center gap-2 flex-wrap">
                  <span class="text-white font-semibold text-sm">${s.fruta}</span>
                  ${getBadgeHTML(s.status, s.segundos_restantes)}
                </div>
                <div class="flex items-center gap-4 mt-1.5 flex-wrap">
                  <span class="text-xs text-gray-500">📍 ${s.setor}</span>
                  <span class="text-xs text-gray-500">🕐 ${formatTimestamp(s.timestamp)}</span>
                  <span class="text-xs text-green-400 font-medium">🏷️ -${s.desconto_sugerido}% sugerido</span>
                </div>
                <div class="flex items-center gap-2 mt-2">
                  <span class="text-xs text-gray-500">IA:</span>
                  <div class="confidence-bar-bg">
                    <div class="confidence-bar-fill" style="width:${s.confianca}%"></div>
                  </div>
                  <span class="text-xs text-gray-400 font-medium">${s.confianca}%</span>
                </div>
              </div>
            </div>

            <div class="flex flex-col gap-2 shrink-0">
              ${!s.respondido ? `
                <button
                  class="btn-action btn-desconto"
                  onclick="responder('${s.id}', 'EXECUTADA')">
                  ✅ Aplicar Desconto
                </button>
                <button
                  class="btn-action btn-ignorar"
                  onclick="responder('${s.id}', 'DESCARTADA')">
                  ✗ Ignorar
                </button>
              ` : `
                <span class="text-xs text-gray-600 text-right">
                  Respondido em<br>${s.tempo_resposta}s
                </span>
              `}
            </div>

          </div>
        </div>
      `).join("");
    }

    // --- Métricas ---

    function renderMetricas(metricas) {
      document.getElementById("metric-total").textContent = metricas.total_deteccoes;
      document.getElementById("metric-alertas").textContent = metricas.alertas_pendentes;
      document.getElementById("metric-efetividade").textContent = metricas.taxa_efetividade + "%";

      // Mini gráfico de barras (desperdício evitado simulado)
      const container = document.getElementById("mini-grafico");
      const alturas = [30, 40, 35, 55, 60, 50, 70, 75];
      container.innerHTML = alturas.map(h => `
        <div style="flex:1; display:flex; align-items:flex-end; height:40px;">
          <div style="width:100%; height:${h}%; background:rgba(52,211,153,0.4); border-radius:3px;"></div>
        </div>
      `).join("");

      // Desperdício simulado baseado em executadas
      const exec = (metricas.total_deteccoes - 137);
      document.getElementById("desperdicio-kg").textContent = (exec * 0.48).toFixed(1);
    }

    // --- Log Terminal ---

    async function atualizarLog() {
      try {
        const res = await fetch(`${API_BASE}/api/log`);
        const data = await res.json();
        const terminal = document.getElementById("log-terminal");
        terminal.innerHTML = data.logs.map(l => `<div>${l}</div>`).join("");
        terminal.scrollTop = terminal.scrollHeight;
      } catch (e) {
        // silencioso
      }
    }

    // --- Fetch principal ---

    async function buscarDados() {
      try {
        const res = await fetch(`${API_BASE}/api/sugestoes`);
        if (!res.ok) throw new Error("Erro na API");
        const data = await res.json();

        dadosAtuais = data.sugestoes;
        renderFeed(data.sugestoes);
        renderMetricas(data.metricas);

        const agora = new Date().toLocaleTimeString("pt-BR");
        document.getElementById("last-update").textContent = `Atualizado às ${agora}`;

      } catch (err) {
        document.getElementById("last-update").textContent = "❌ Sem conexão com o servidor";
      }
    }

    // --- Ação do Gerente ---

    async function responder(id, acao) {
      try {
        const res = await fetch(`${API_BASE}/api/responder`, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ id, acao })
        });

        const data = await res.json();

        if (!res.ok) {
          showToast(data.erro || "Erro ao registrar ação.", "error");
          return;
        }

        const msg = acao === "EXECUTADA"
          ? `✅ Desconto aplicado! Respondido em ${data.tempo_resposta_segundos}s`
          : `— Sugestão ignorada e removida da fila.`;
        showToast(msg);

        await buscarDados();

      } catch (err) {
        showToast("Falha na comunicação com o servidor.", "error");
      }
    }

    // --- Inicialização ---

    buscarDados();
    atualizarLog();
    setInterval(buscarDados, 5000);
    setInterval(atualizarLog, 7000);
  </script>

</body>
</html>""")